# Pre-Phase Notebook: Text Extraction
Optional helper notebook to convert raw ZDF archive PDFs into canonical `.pdf_episodes.txt` dumps in `data/01_input/zdf_archive`.

Primary workflow remains text-dump-first; this notebook is only needed when raw PDFs are the available source.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [ ]:
from process.bootstrap import bootstrap_notebook_paths

ROOT, SRC = bootstrap_notebook_paths()
ROOT

## 2) Input Discovery
Configure expected archive PDF names and inspect what exists in `data/01_input/zdf_archive`.

In [ ]:
from pathlib import Path
import pandas as pd

from process.mention_detection.config import ZDF_ARCHIVE_DIR
from process.text_extraction.text import (
    assemble_episode_blocks_from_pdf,
    load_episode_blocks_from_txt,
    write_episode_text_dump,
)

PDF_NAMES = [
    'Markus Lanz_2008-2010.pdf',
    'Markus Lanz_2011-2015.pdf',
    'Markus Lanz_2016-2020.pdf',
    'Markus Lanz_2021-2024.pdf',
]

archive_dir = ROOT / ZDF_ARCHIVE_DIR
pdf_paths = [archive_dir / name for name in PDF_NAMES]

existing_pdfs = [p for p in pdf_paths if p.exists()]
missing_pdfs = [p for p in pdf_paths if not p.exists()]

pd.DataFrame({
    'pdf': [p.name for p in pdf_paths],
    'exists': [p.exists() for p in pdf_paths],
})

## 3) Convert Available PDFs To Canonical Text Dumps
Writes `*.pdf_episodes.txt` files next to each source PDF.

Safety default: existing outputs are not overwritten unless `overwrite_existing = True`.

In [ ]:
overwrite_existing = False
conversion_results = []

for pdf_path in existing_pdfs:
    output_path = pdf_path.with_name(f"{pdf_path.name}_episodes.txt")

    if output_path.exists() and not overwrite_existing:
        conversion_results.append({
            'pdf': pdf_path.name,
            'output': output_path.name,
            'episodes': None,
            'status': 'skipped_existing',
        })
        continue

    episode_blocks = assemble_episode_blocks_from_pdf(pdf_path)
    write_episode_text_dump(episode_blocks, output_path)

    conversion_results.append({
        'pdf': pdf_path.name,
        'output': output_path.name,
        'episodes': len(episode_blocks),
        'status': 'written',
    })

if missing_pdfs:
    print('Missing PDFs:')
    for p in missing_pdfs:
        print(' -', p.name)

pd.DataFrame(conversion_results)

## 4) Optional Roundtrip Check
Reads one generated dump with the normal splitter to verify parseability.

In [ ]:
written_rows = [row for row in conversion_results if row['status'] == 'written']

if not written_rows:
    'No new dump files written in this run.'
else:
    sample = written_rows[0]
    sample_path = archive_dir / sample['output']
    sample_blocks = load_episode_blocks_from_txt(sample_path)
    {
        'sample_output': sample_path.name,
        'parsed_episode_blocks': len(sample_blocks),
    }